# S6E10 | RealMLP Synthesis-View Rider (GPU) — new information for the stack

Trains a **RealMLP (pytabkit)** on the *synthesis view*: Arseniy Shutko original-model teacher logits
(+0.000287 measured), empirical lookup rates (+0.000927 measured), route-profile geometry, structural
zero flags — with the **original 129,880 rows appended to every fold-train** (+0.001029 vs +0.000927, USAGE.md).

- Folds: `StratifiedKFold(5, shuffle=True, random_state=42)` — the shared public folds → drop-in OOF+test pair.
- Architecture credit: Vladimir Demidov (yekenot) `PS|S6|E10: RealMLP · PyTabKit`.
- Data credit: goodpjw2008 (members OOF vault + Probe G anchor), dariushafshar (original-data-aligned), Arseniy Shutko (original survey).

Outputs: `oof_rider_synthview.npy`, `test_rider_synthview.npy`, `rider_ledger.json`,
blend grid `submission_rider_w*.csv`, primary `submission.csv` (anchor 0.94 + rider 0.06).

In [ ]:

import os, sys, gc, time, glob, random, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
SMOKE = os.environ.get('S6E10_SMOKE', '0') == '1'
FOLDS = 2 if SMOKE else 5
N_ENS = 1 if SMOKE else 4
N_EPOCHS = 1 if SMOKE else 4
ROOTS = os.environ.get('INPUT_ROOTS', '/kaggle/input').split(':')

def find_file(name, must=True):
    for root in ROOTS:
        hits = sorted(glob.glob(f'{root}/**/{name}', recursive=True))
        if hits:
            return hits[0]
    if must:
        raise FileNotFoundError(name)
    return None

print('SMOKE =', SMOKE, '| FOLDS', FOLDS, '| N_ENS', N_ENS, '| N_EPOCHS', N_EPOCHS)
try:
    import pytabkit
    from importlib.metadata import version
    print('pytabkit', version('pytabkit'))
except ImportError:
    print('installing pytabkit ...')
    ok = os.system('pip install -q pytabkit')
    if ok != 0:
        whl = find_file('pytabkit*.whl')
        print('internet install failed, falling back to wheelhouse:', whl)
        os.system(f'pip install -q --no-index --find-links={os.path.dirname(whl)} pytabkit')
    from pytabkit import RealMLP_TD_Classifier
else:
    from pytabkit import RealMLP_TD_Classifier
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())


In [ ]:

t0 = time.time()
train = pd.read_csv(find_file('playground-series-s6e10/train.csv'))
test = pd.read_csv(find_file('playground-series-s6e10/test.csv'))
if SMOKE:
    train = train.iloc[:30000].copy(); test = test.iloc[:12000].copy()
print('train', train.shape, '| test', test.shape)

SURVEY = ["Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking",
          "Gate location", "Food and drink", "Online boarding", "Seat comfort",
          "Inflight entertainment", "On-board service", "Leg room service", "Baggage handling",
          "Checkin service", "Cleanliness"]
CATS = ["Gender", "Customer Type", "Type of Travel", "Class"]
FD = 'Flight Distance'
TARGET = 'satisfaction'

# ---- original rows (append arm) ----
orig = pd.read_csv(find_file('original_aligned.csv'))
orig = orig.rename(columns={c: c for c in orig.columns})
if SMOKE:
    orig = orig.iloc[:8000].copy()
print('orig', orig.shape, '| cols', list(orig.columns)[:8], '...')

# ---- lookup tables (21 empirical-rate columns, +0.000927 measured) ----
D_ALIGN = os.path.dirname(find_file('lookup_long.csv'))
sys.path.append(D_ALIGN)
L = pd.read_csv(os.path.join(D_ALIGN, 'lookup_long.csv'), dtype={'value': str}, keep_default_na=False)

def add_lookup(df):
    # unseen levels on comp data -> fill with the feature's empirical rate mean
    for f, t in L[L.feature.isin(df.columns)].groupby('feature'):
        col = df[f].astype(str).fillna('nan').map(dict(zip(t.value, t.rate)))
        n_na = int(col.isna().sum())
        if n_na:
            print(f'  lookup {f}: {n_na} unseen levels -> feature rate mean')
            col = col.fillna(float(t['rate'].mean()))
        df['orig_rate__' + f] = col.astype(np.float32)
    # the combination key (Type of Travel|Class|Customer Type), from USAGE.md
    if 'Type of Travel' in df.columns and not any(c.startswith('orig_rate__Type of Travel|') for c in df.columns):
        key = df['Type of Travel'].astype(str) + '|' + df['Class'].astype(str) + '|' + df['Customer Type'].astype(str)
        row = L[L.feature == 'Type of Travel|Class|Customer Type']
        if len(row):
            col = key.map(dict(zip(row.value, row.rate)))
            n_na = int(col.isna().sum())
            if n_na:
                print(f'  lookup combo: {n_na} unseen levels -> rate mean')
                col = col.fillna(float(row['rate'].mean()))
            df['orig_rate__tot_cls_cust'] = col.astype(np.float32)
    return df

# ---- original-model teacher logits (+0.000287 measured) ----
from original_model import score as score_original

# ---- route geometry on train+test+orig (target-free) ----
allf = pd.concat([train.drop(columns=['id', TARGET], errors='ignore'),
                  test.drop(columns=['id'], errors='ignore'),
                  orig.drop(columns=['id', TARGET], errors='ignore')], ignore_index=True)
g = allf.groupby(FD)
agg = g[SURVEY + ['Age']].mean().add_prefix('fd_mean_').astype(np.float32)
agg['fd_count'] = g.size().astype(np.int32)
top_class = allf['Class'].mode()[0]
agg['fd_share_biz'] = (allf['Class'] == top_class).groupby(allf[FD]).mean().astype(np.float32)
del allf; gc.collect()

def build(df, has_target):
    df = add_lookup(df)
    ogp = score_original(df)
    df['ogp_xgb_d8'] = ogp['ogp_xgb_d8'].values.astype(np.float32)
    df['ogp_lgb'] = ogp['ogp_lgb'].values.astype(np.float32)
    df = df.join(agg, on=FD)
    for c in ['Arrival Delay in Minutes', 'Departure Delay in Minutes']:
        df[c] = df[c].fillna(0.0)
    df['log_dep_delay'] = np.log1p(np.maximum(df['Departure Delay in Minutes'], 0)).astype(np.float32)
    df['log_arr_delay'] = np.log1p(np.maximum(df['Arrival Delay in Minutes'], 0)).astype(np.float32)
    df['delay_diff'] = (df['Arrival Delay in Minutes'] - df['Departure Delay in Minutes']).astype(np.float32)
    df['has_no_delay'] = ((df['Departure Delay in Minutes'] == 0) & (df['Arrival Delay in Minutes'] == 0)).astype(np.int32)
    for c in SURVEY:
        df[f'{c}_is_zero'] = (df[c] == 0).astype(np.int32)
    df['wifi0_booking0'] = ((df['Inflight wifi service'] == 0) & (df['Ease of Online booking'] == 0)).astype(np.int32)
    df['total_survey'] = df[SURVEY].sum(axis=1).astype(np.float32)
    df['zero_survey_count'] = (df[SURVEY] == 0).sum(axis=1).astype(np.int32)
    df['fd_len'] = df[FD].astype(str).str.len().astype(np.int32)
    df['fd_mod100'] = (df[FD] % 100).astype(np.int32)
    df['age_mod10'] = (df['Age'] % 10).astype(np.int32)
    for c in CATS:
        df[c] = df[c].astype('category').cat.codes.astype(np.int32)
    drop = ['id'] + ([TARGET] if has_target else [])
    return df

tr_df = build(train, True);  y = train[TARGET].astype(np.int8).values
te_df = build(test, False)
og_df = build(orig, True);   y_og = orig[TARGET].astype(np.int8).values

DROP = {'id', TARGET}
FEATS = [c for c in tr_df.columns if c not in DROP]
X = tr_df[FEATS].astype(np.float32); Xt = te_df[FEATS].astype(np.float32); Xo = og_df[FEATS].astype(np.float32)
for name, M in [('train', X), ('test', Xt), ('orig', Xo)]:
    n_nan = int(M.isna().sum().sum())
    if n_nan:
        bad = M.isna().sum(); print(f'  {name}: {n_nan} NaNs in', bad[bad > 0].index.tolist()[:8], '-> fill 0')
        M.fillna(0.0, inplace=True)
print(f'FE done in {time.time()-t0:.1f}s | features: {len(FEATS)}')
print(FEATS)


In [ ]:

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

params = dict(
    random_state=42, verbosity=1, val_metric_name='1-auc_ovr',
    n_ens=N_ENS, n_epochs=N_EPOCHS, batch_size=288,
    use_early_stopping=False,
    lr=0.053, wd=0.015, sq_mom=0.988,
    lr_sched='flat_anneal', wd_sched='cos_log_15',
    first_layer_lr_factor=0.25,
    embedding_size=5, max_one_hot_cat_size=2,
    hidden_sizes=[512, 256, 128], act='silu',
    p_drop=0.05, p_drop_sched='pow5',
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name='gelu',
    plr_lr_factor=0.1151, plr_sigma=0.75,
    ls_eps=0.01, ls_eps_sched='sqrt_cos',
    add_front_scale=False, bias_init_mode='neg-uniform-dynamic-2',
    tfms=['one_hot', 'median_center', 'robust_scale', 'smooth_clip', 'embedding', 'l2_normalize'],
)

oof_rider = np.zeros(len(X), dtype=np.float64)
test_rider = np.zeros(len(Xt), dtype=np.float64)
fold_aucs = []
t_ride = time.time()

for fold, (a, b) in enumerate(skf.split(X, y)):
    tf0 = time.time()
    # row-append arm: fold-train + original rows (USAGE.md: +0.001029 w/ lookups vs +0.000927)
    X_tr = pd.concat([X.iloc[a], Xo], ignore_index=True)
    y_tr = np.concatenate([y[a], y_og])
    X_va = X.iloc[b]
    model = RealMLP_TD_Classifier(**params)
    model.fit(X_tr, y_tr, X_va, y[b])
    oof_rider[b] = model.predict_proba(X_va)[:, 1]
    test_rider += model.predict_proba(Xt)[:, 1] / FOLDS
    auc_f = roc_auc_score(y[b], oof_rider[b])
    fold_aucs.append(float(auc_f))
    print(f'fold {fold+1}/{FOLDS} AUC {auc_f:.6f} ({time.time()-tf0:.0f}s)')
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    gc.collect()

oof_auc = roc_auc_score(y, oof_rider)
print(f'==> RIDER OOF AUC: {oof_auc:.6f} | total {time.time()-t_ride:.0f}s')
np.save('oof_rider_synthview.npy', oof_rider)
np.save('test_rider_synthview.npy', test_rider)
with open('rider_ledger.json', 'w') as f:
    json.dump({'oof_auc': float(oof_auc), 'fold_aucs': fold_aucs,
               'folds': FOLDS, 'n_ens': N_ENS, 'n_epochs': N_EPOCHS,
               'features': FEATS, 'params': {k: str(v) for k, v in params.items()}}, f, indent=2)


In [ ]:

def pr(x):
    return (pd.Series(x).rank().values - 0.5) / len(x)

def to_logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

# ---- anchor: gpjw Probe G (public LB 0.96216) ----
probeG = pd.read_csv(find_file('meta_part6_probeG_096216_1028.csv'))
if len(probeG) != len(test):
    print(f'SMOKE: truncating probeG {len(probeG)} -> {len(test)}')
    probeG = probeG.iloc[:len(test)].copy()
assert (probeG['id'].values == test['id'].values).all()
z_anchor = to_logit(pr(probeG['satisfaction'].values))
z_rider = to_logit(pr(test_rider))

ids = test['id'].values
grid = [0.02, 0.04, 0.06, 0.08, 0.12] if not SMOKE else [0.5]
for w in grid:
    z = (1 - w) * z_anchor + w * z_rider
    out = pd.DataFrame({'id': ids, 'satisfaction': pr(z)})
    out.to_csv(f'submission_rider_w{w:.2f}.csv', index=False)
    print(f'wrote submission_rider_w{w:.2f}.csv')

# PRIMARY auto-submit: w=0.06 (mirrors V15's 6% new-info architecture)
w = 0.06
z = (1 - w) * z_anchor + w * z_rider
sub = pd.DataFrame({'id': ids, 'satisfaction': pr(z)})
assert len(sub) == len(test) and sub['satisfaction'].notna().all()
assert (sub['id'] == ids).all()
sub.to_csv('submission.csv', index=False)
print('PRIMARY submission.csv written (anchor 0.94 + rider 0.06)')

# rider-only reference file
pd.DataFrame({'id': ids, 'satisfaction': pr(test_rider)}).to_csv('submission_rider_only.csv', index=False)
from scipy.stats import spearmanr
print('Spearman(rider, probeG) =', spearmanr(pr(test_rider), pr(probeG['satisfaction'].values)).statistic)
print('rider test mean/std:', test_rider.mean(), test_rider.std())
